In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import json

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def find_project_root(marker=".venv"):
    current_path = Path.cwd().resolve()
    for parent in [current_path] + list(current_path.parents):
        if (parent / marker).is_dir():
            return parent
    raise FileNotFoundError(f"Project root marker '{marker}' not found.")

PROJECT_ROOT = find_project_root()
INPUT_DIR = PROJECT_ROOT / "input_data"
OUTPUT_FILE = PROJECT_ROOT / "output_data" / "master_output.csv"
CONFIG_FILE = PROJECT_ROOT / "config" / "master.json"

TARGET_COLS = ["engine_temp", "rpm", "vehicle_speed", "fuel_pressure"]
VALIDITY_THRESHOLD = 0.10
TRIP_GAP_HOURS = 8

STD_LAT = "GPS Latitude"
STD_LON = "GPS Longitude"
STD_TIME = "GPS Time"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
CONFIG_FILE.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
valid_dfs = []
ALL_NUMERIC_TARGETS = TARGET_COLS + [STD_LAT, STD_LON]

for file_path in INPUT_DIR.glob("*.csv"):
    try:
        def usecols_filter(col_name):
            if col_name in TARGET_COLS:
                return True
            col_lower = col_name.lower()
            if "gps latitude" in col_lower or "gps longitude" in col_lower or "gps time" in col_lower:
                return True
            return False

        df = pd.read_csv(file_path, usecols=usecols_filter)
        actual_cols = df.columns.tolist()
        
        lat_col = next((c for c in actual_cols if "gps latitude" in c.lower()), None)
        lon_col = next((c for c in actual_cols if "gps longitude" in c.lower()), None)
        time_col = next((c for c in actual_cols if "gps time" in c.lower()), None)

        missing_targets = list(set(TARGET_COLS) - set(actual_cols))
        missing_report = missing_targets.copy()
        
        if not lat_col: 
            missing_report.append("GPS Latitude (substring match)")
        if not lon_col: 
            missing_report.append("GPS Longitude (substring match)")
        if not time_col: 
            missing_report.append("GPS Time (substring match)")

        if missing_report:
            logging.warning(f"File {file_path.name} skipped. Missing columns: {missing_report}")
            continue

        df = df.rename(columns={
            lat_col: STD_LAT, 
            lon_col: STD_LON, 
            time_col: STD_TIME
        })

        initial_rows = len(df)
        
        df[STD_TIME] = df[STD_TIME].astype(str).str.replace("GMT", "", regex=False)
        df[STD_TIME] = pd.to_datetime(df[STD_TIME], format="%a %b %d %H:%M:%S %z %Y", errors='coerce')
        
        valid_time_df = df.dropna(subset=[STD_TIME]).copy()
        dropped_rows = initial_rows - len(valid_time_df)
        
        if dropped_rows > 0:
            logging.info(f"File {file_path.name}: Dropped {dropped_rows} rows due to invalid/missing {STD_TIME}.")
            
        if valid_time_df.empty:
            logging.warning(f"File {file_path.name} is empty after dropping invalid time rows. Skipped.")
            continue

        for col in ALL_NUMERIC_TARGETS:
            valid_time_df[col] = pd.to_numeric(valid_time_df[col], errors='coerce')

        total_rows = len(valid_time_df)
        threshold = total_rows * VALIDITY_THRESHOLD
        passed = True

        for col in ALL_NUMERIC_TARGETS:
            valid_count = valid_time_df[col].notna().sum()
            if valid_count < threshold:
                logging.warning(f"File {file_path.name} failed threshold on {col}. Valid: {valid_count}/{total_rows}. Skipped.")
                passed = False
                break

        if passed:
            valid_dfs.append(valid_time_df)
            logging.info(f"File {file_path.name} passed all checks.")

    except Exception as e:
        logging.error(f"Error processing {file_path.name}: {e}")

In [ ]:
if not valid_dfs:
    logging.error("No valid dataframes to merge. Pipeline halted.")
else:
    master_df = pd.concat(valid_dfs, ignore_index=True)

    master_df[STD_TIME] = master_df[STD_TIME].dt.tz_convert('UTC')
    master_df = master_df.sort_values(by=STD_TIME, ascending=True).reset_index(drop=True)
    
    time_diffs = master_df[STD_TIME].diff()
    master_df['Trip Number'] = (time_diffs > pd.Timedelta(hours=TRIP_GAP_HOURS)).cumsum() + 1
    
    master_df[STD_TIME] = master_df[STD_TIME].dt.strftime('%Y-%m-%d %H:%M:%S') + '+00:00'

    master_df.to_csv(OUTPUT_FILE, index=False)
    logging.info(f"Master CSV written to {OUTPUT_FILE} with {len(master_df)} rows.")

    schema = {
        "metadata": {
            "file_path": str(OUTPUT_FILE.resolve()),
            "total_rows": int(len(master_df)),
            "total_trips": int(master_df['Trip Number'].max()),
            "time_column_format": "YYYY-MM-DD HH:MM:SS+00:00 (UTC)",
            "trip_gap_threshold_hours": TRIP_GAP_HOURS
        },
        "schema": {}
    }
    
    for col in master_df.columns:
        dtype_str = str(master_df[col].dtype)
        if 'int' in dtype_str:
            mapped_type = 'integer'
        elif 'float' in dtype_str:
            mapped_type = 'float'
        elif col == STD_TIME:
            mapped_type = 'datetime string'
        else:
            mapped_type = 'string'
            
        schema["schema"][col] = {
            "type": mapped_type,
            "nullable": bool(master_df[col].isnull().any())
        }

    with open(CONFIG_FILE, 'w') as f:
        json.dump(schema, f, indent=4)
    
    logging.info(f"Pipeline complete. Master schema JSON written to {CONFIG_FILE}.")